In [ ]:
!pip install kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahdimashayekhi/disease-risk-from-daily-habits")

print("Path to dataset files:", path)

100%|██████████| 20.8M/20.8M [00:00<00:00, 58.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mahdimashayekhi/disease-risk-from-daily-habits/versions/1


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.9 MB/s eta 0:00:00


In [ ]:
#import library
import pandas as pd
import os

# Lihat isi folder dataset
print(os.listdir(path))

['health_lifestyle_classification.csv']


In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

In [ ]:
df = pd.read_csv(path + "/health_lifestyle_classification.csv")
df.head()

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,...,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,...,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,...,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,...,High,1,NaN,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,...,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   survey_code               100000 non-null  int64  
 1   age                       100000 non-null  int64  
 2   gender                    100000 non-null  object 
 3   height                    100000 non-null  float64
 4   weight                    100000 non-null  float64
 5   bmi                       100000 non-null  float64
 6   bmi_estimated             100000 non-null  float64
 7   bmi_scaled                100000 non-null  float64
 8   bmi_corrected             100000 non-null  float64
 9   waist_size                100000 non-null  float64
 10  blood_pressure            92331 non-null   float64
 11  heart_rate                85997 non-null   float64
 12  cholesterol               100000 non-null  float64
 13  glucose                   100000 non-null  fl

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,sleep_quality,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,alcohol_consumption,smoking_level,water_intake,screen_time,stress_level,mental_health_score,mental_health_support,education_level,job_type,occupation,income,diet_type,exercise_type,device_usage,healthcare_access,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,118.264254,60.749825,214.580523,103.008176,NaN,6.475885,Fair,7.671313,0.356918,13320.942595,2673.546960,44.476887,NaN,Non-smoker,1.694262,5.003963,2,8,No,PhD,Tech,Farmer,6759.821719,Vegan,Strength,High,Poor,No,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,117.917986,66.463696,115.794002,116.905134,10.131597,8.428410,Good,9.515198,0.568219,11911.201401,2650.376972,74.663405,Regularly,Light,0.716409,5.925455,3,9,No,High School,Office,Engineer,6240.517690,Vegan,Cardio,Moderate,Moderate,No,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,123.073698,76.043212,138.134787,89.180302,NaN,5.702164,Poor,5.829853,3.764406,2974.035375,1746.755144,19.702382,Regularly,Heavy,2.487900,4.371250,0,1,No,Master,Office,Teacher,3429.179266,Vegan,Cardio,High,Good,Yes,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,148.173453,68.781981,203.017447,128.375798,18.733179,5.188316,Good,9.489693,0.889474,5321.539497,2034.193242,82.580050,Occasionally,Heavy,2.643335,4.116064,10,4,No,Master,Labor,Teacher,2618.503534,Vegetarian,Mixed,Low,Moderate,No,High,1,NaN,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,150.613181,92.335358,200.412439,94.813332,16.038701,7.912514,Good,7.275450,2.901608,9791.376712,2386.210257,45.961322,NaN,Heavy,1.968393,3.180087,9,7,Yes,Master,Unemployed,Doctor,3662.086276,Vegan,NaN,Low,Moderate,Yes,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy


In [ ]:
df.isnull().sum()

,0
survey_code,0
age,0
gender,0
height,0
weight,0
bmi,0
bmi_estimated,0
bmi_scaled,0
bmi_corrected,0
waist_size,0


In [ ]:
df = df.drop(columns=[
'survey_code',
'bmi_estimated','bmi_scaled','bmi_corrected',
'mental_health_score','mental_health_support',
'education_level','job_type','occupation',
'income','healthcare_access','insurance',
'sunlight_exposure','family_history','pet_owner',
'electrolyte_level','gene_marker_flag',
'environmental_risk_score','daily_supplement_dosage'
])

In [ ]:
# Kolom numerik
num_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns

for col in num_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

# Kolom kategorikal
cat_cols = df.select_dtypes(
    include='object'
).columns

for col in cat_cols:
    df[col] = df[col].fillna(
        "Unknown"
    )

In [ ]:
df.isnull().sum()

,0
age,0
gender,0
height,0
weight,0
bmi,0
waist_size,0
blood_pressure,0
heart_rate,0
cholesterol,0
glucose,0


In [ ]:
df.head()

,age,gender,height,weight,bmi,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,sleep_quality,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,alcohol_consumption,smoking_level,water_intake,screen_time,stress_level,diet_type,exercise_type,device_usage,meals_per_day,caffeine_intake,target
0,56,Male,173.416872,56.886640,18.915925,72.165130,118.264254,60.749825,214.580523,103.008176,14.983414,6.475885,Fair,7.671313,0.356918,13320.942595,2673.546960,44.476887,Unknown,Non-smoker,1.694262,5.003963,2,Vegan,Strength,High,5,Moderate,healthy
1,69,Female,163.207380,97.799859,36.716278,85.598889,117.917986,66.463696,115.794002,116.905134,10.131597,8.428410,Good,9.515198,0.568219,11911.201401,2650.376972,74.663405,Regularly,Light,0.716409,5.925455,3,Vegan,Cardio,Moderate,5,High,healthy
2,46,Male,177.281966,80.687562,25.673050,90.295030,123.073698,76.043212,138.134787,89.180302,14.983414,5.702164,Poor,5.829853,3.764406,2974.035375,1746.755144,19.702382,Regularly,Heavy,2.487900,4.371250,0,Vegan,Cardio,High,4,Moderate,healthy
3,32,Female,172.101255,63.142868,21.318480,100.504211,148.173453,68.781981,203.017447,128.375798,18.733179,5.188316,Good,9.489693,0.889474,5321.539497,2034.193242,82.580050,Occasionally,Heavy,2.643335,4.116064,10,Vegetarian,Mixed,Low,1,Unknown,healthy
4,60,Female,163.608816,40.000000,14.943302,69.021150,150.613181,92.335358,200.412439,94.813332,16.038701,7.912514,Good,7.275450,2.901608,9791.376712,2386.210257,45.961322,Unknown,Heavy,1.968393,3.180087,9,Vegan,Unknown,Low,1,High,healthy


In [ ]:
# ============================================================
# FEATURE ENGINEERING
# Menggunakan nama kolom asli dataset
# ============================================================

# ------------------------------------------------------------
# 1. BMI (Koreksi dan Optimasi)
# ------------------------------------------------------------
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# ------------------------------------------------------------
# 2. Physical Activity
# Kombinasi aktivitas fisik dan langkah harian
# ------------------------------------------------------------
df['physical_activity'] = (
    df['physical_activity'] +
    (df['daily_steps'] / 1000)
) / 2

# ------------------------------------------------------------
# 3. Sleep Quality
# Menyesuaikan kualitas tidur berdasarkan durasi tidur
# ------------------------------------------------------------
df['sleep_quality'] = np.where(
    df['sleep_hours'] < 5, 'Poor',
    np.where(
        df['sleep_hours'] < 7, 'Average',
        np.where(
            df['sleep_hours'] < 9, 'Good',
            'Excellent'
        )
    )
)

# ------------------------------------------------------------
# 4. Stress Level
# Penyesuaian berdasarkan work hours dan screen time
# ------------------------------------------------------------
df['stress_level'] = (
    df['stress_level'] +
    (df['work_hours'] / 10) +
    (df['screen_time'] / 5)
) / 3

# ------------------------------------------------------------
# 5. Calorie Intake
# Penyesuaian berdasarkan aktivitas
# ------------------------------------------------------------
df['calorie_intake'] = (
    df['calorie_intake'] +
    (df['physical_activity'] * 100)
)

# ------------------------------------------------------------
# 6. Sugar Intake
# Penyesuaian berdasarkan jumlah makan
# ------------------------------------------------------------
df['sugar_intake'] = (
    df['sugar_intake'] /
    (df['meals_per_day'] + 1)
)

# ------------------------------------------------------------
# 7. Water Intake
# Disesuaikan dengan berat badan
# ------------------------------------------------------------
df['water_intake'] = (
    df['water_intake'] +
    (df['weight'] / 20)
) / 2

# ------------------------------------------------------------
# 8. Heart Rate
# Koreksi sederhana berdasarkan stress dan aktivitas
# ------------------------------------------------------------
df['heart_rate'] = (
    df['heart_rate'] +
    df['stress_level'] -
    (df['physical_activity'] / 2)
)

# ------------------------------------------------------------
# 9. Blood Pressure
# Penyesuaian berdasarkan usia dan BMI
# ------------------------------------------------------------
df['blood_pressure'] = (
    df['blood_pressure'] +
    (df['age'] / 5) +
    (df['bmi'] / 2)
) / 3

# ------------------------------------------------------------
# 10. Glucose
# Penyesuaian berdasarkan sugar intake
# ------------------------------------------------------------
df['glucose'] = (
    df['glucose'] +
    (df['sugar_intake'] * 0.5)
)

# ------------------------------------------------------------
# 11. Cholesterol
# Penyesuaian berdasarkan BMI
# ------------------------------------------------------------
df['cholesterol'] = (
    df['cholesterol'] +
    (df['bmi'] * 2)
)

# ------------------------------------------------------------
# 12. Insulin
# Penyesuaian berdasarkan glucose
# ------------------------------------------------------------
df['insulin'] = (
    df['insulin'] +
    (df['glucose'] / 10)
)

# ------------------------------------------------------------
# 13. Waist Size
# Estimasi ulang berdasarkan BMI
# ------------------------------------------------------------
df['waist_size'] = (
    df['waist_size'] +
    (df['bmi'] * 1.5)
) / 2

print("Feature engineering selesai!")
print("Jumlah kolom:", df.shape[1])

Feature engineering selesai!
Jumlah kolom: 29


In [ ]:
df.head()

,age,gender,height,weight,bmi,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,sleep_quality,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,alcohol_consumption,smoking_level,water_intake,screen_time,stress_level,diet_type,exercise_type,device_usage,meals_per_day,caffeine_intake,target
0,56,Male,173.416872,56.886640,18.915925,50.269509,46.307406,58.586334,252.412373,106.714583,25.654873,6.475885,Average,7.671313,6.838930,13320.942595,3357.440008,7.412815,Unknown,Non-smoker,2.269297,5.003963,1.255975,Vegan,Strength,High,5,Moderate,healthy
1,69,Female,163.207380,97.799859,36.716278,70.336653,50.025375,65.056045,189.226557,123.127084,22.444306,8.428410,Good,9.515198,6.239710,11911.201401,3274.348008,12.443901,Regularly,Light,2.803201,5.925455,1.712204,Vegan,Cardio,Moderate,5,High,healthy
2,46,Male,177.281966,80.687562,25.673050,64.402303,48.370074,74.844347,189.480888,91.150541,24.098468,5.702164,Average,5.829853,3.369221,2974.035375,2083.677224,3.940476,Regularly,Heavy,3.261139,4.371250,0.485745,Vegan,Cardio,High,4,Moderate,healthy
3,32,Female,172.101255,63.142868,21.318480,66.240965,55.077564,71.153288,245.654407,149.020810,33.635260,5.188316,Average,9.489693,3.105507,5321.539497,2344.743931,41.290025,Occasionally,Heavy,2.900239,4.116064,3.924061,Vegetarian,Mixed,Low,1,Unknown,healthy
4,60,Female,163.608816,40.000000,14.943302,45.718052,56.694944,92.616633,230.299044,106.303662,26.669068,7.912514,Good,7.275450,6.346492,9791.376712,3020.859499,22.980661,Unknown,Heavy,1.984196,3.180087,3.454521,Vegan,Unknown,Low,1,High,healthy


In [ ]:
# ============================================================
# TRAIN-TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Ubah target menjadi numerik
df['target'] = df['target'].map({
    'healthy': 0,
    'diseased': 1
})

# Pisahkan fitur dan target
X = df.drop('target', axis=1)
y = df['target']

# Deteksi kolom kategorikal otomatis
cat_features = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("Jumlah fitur:", X.shape[1])
print("Jumlah fitur kategorikal:", len(cat_features))
print("Fitur kategorikal:", cat_features)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Jumlah fitur: 28
Jumlah fitur kategorikal: 8
Fitur kategorikal: ['gender', 'sleep_quality', 'alcohol_consumption', 'smoking_level', 'diet_type', 'exercise_type', 'device_usage', 'caffeine_intake']


In [ ]:
cat_features = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

# Ubah semua kolom kategorikal menjadi string
for col in cat_features:
    X[col] = X[col].astype(str)

In [ ]:
# ============================================================
# MODELING CATBOOST
# ============================================================

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=10,
    random_strength=3,
    auto_class_weights='Balanced',
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_state=42,
    verbose=200
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=300
)

0:	learn: 0.5071340	test: 0.5035735	best: 0.5035735 (0)	total: 337ms	remaining: 16m 50s
200:	learn: 0.5545486	test: 0.5008936	best: 0.5060793 (5)	total: 42.8s	remaining: 9m 56s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.5060793149
bestIteration = 5

Shrink model to first 6 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, eval_metric='Accuracy', iterations=3000, l2_leaf_reg=10, learning_rate=0.03, loss_function='Logloss', random_state=42, random_strength=3, verbose=200)

In [ ]:
# ============================================================
# PREDIKSI DAN EVALUASI
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.51185

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.52      0.60     14019
           1       0.30      0.49      0.38      5981

    accuracy                           0.51     20000
   macro avg       0.51      0.51      0.49     20000
weighted avg       0.59      0.51      0.53     20000


Confusion Matrix:
[[7296 6723]
 [3040 2941]]


In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.get_feature_importance()
}).sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance.head(20))

                Feature  Importance
15          daily_steps   16.505261
1                gender   15.058803
8           cholesterol   14.904825
13           work_hours   14.800407
17         sugar_intake   11.386899
16       calorie_intake    7.244047
26        meals_per_day    7.093403
24        exercise_type    5.102906
12        sleep_quality    4.918271
27      caffeine_intake    2.074884
4                   bmi    0.740692
18  alcohol_consumption    0.169602
11          sleep_hours    0.000000
10              insulin    0.000000
9               glucose    0.000000
7            heart_rate    0.000000
5            waist_size    0.000000
6        blood_pressure    0.000000
2                height    0.000000
3                weight    0.000000


In [ ]:
# ============================================================
# HYPERPARAMETER TUNING (GRID SEARCH)
# ============================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    'depth': [6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05],
    'l2_leaf_reg': [3, 5, 10],
    'iterations': [1000, 2000]
}

base_model = CatBoostClassifier(
    auto_class_weights='Balanced',
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_state=42,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Accuracy:")
print(grid_search.best_score_)

Fitting 3 folds for each of 54 candidates, totalling 162 fits


In [ ]:
# ============================================================
# MODEL TERBAIK HASIL TUNING
# ============================================================

best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Accuracy Setelah Tuning:",
      accuracy_score(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))